
# Agricultural Risk Intelligence

## 1. Objective
This notebook investigates historical agricultural yield anomalies and develops interpretable risk indicators using the existing integrated APY and annual district-level weather dataset.

## 2. Key Questions
- Which district–crop–season combinations show unusually low or high yields?
- How does current yield compare with the historical performance of the same combination?
- Do available weather observations coincide with historical yield anomalies?
- Which indicators are sufficiently supported by data to be considered for application integration?

## 3. Methodology
- Use individual-crop records and exclude aggregate totals from crop-level analysis.
- Establish historical baselines for comparable district–crop–season combinations.
- Identify unusual yield observations using transparent statistical thresholds.
- Examine weather associations only where weather data is available.
- Validate sample sizes, data coverage, and indicator reliability before exporting results.

## 4. Limitations
Historical anomalies are investigation signals, not proof of crop failure or its cause. Annual district-level weather summaries have unverified crop-season alignment and cannot support short-term weather forecasts. Missing observations will be treated as unavailable, not as evidence of low risk.

## 5. Expected Outputs
Risk-indicator tables, validation results, and metadata for potential integration into BharatCrop-Intelligence.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data/processed/weather_regional/apy_weather_integrated_exploratory.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "models/agricultural_risk_intelligence"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)

required_cols = [
    "State", "District", "Crop", "Season",
    "start_year", "Yield", "Crop_Category",
    "weather_available", "tmin_mean_c",
    "tmax_mean_c", "precipitation_total_mm"
]

print("Dataset shape:", df.shape)
print("Year range:", df["start_year"].min(), "to", df["start_year"].max())
print("\nCrop categories:")
print(df["Crop_Category"].value_counts(dropna=False))
print("\nRequired columns:")
print({col: col in df.columns for col in required_cols})

assert all(col in df.columns for col in required_cols), (
    "Required columns are missing. Check the source dataset."
)

print("\nSetup and schema validation passed.")


Dataset shape: (376272, 39)
Year range: 2013 to 2024

Crop categories:
Crop_Category
Individual Crop Candidate                                  226560
Grouping — Verify Source Definition                        107640
Aggregate Total — Exclude from Individual-Crop Analysis     42072
Name: count, dtype: int64

Required columns:
{'State': True, 'District': True, 'Crop': True, 'Season': True, 'start_year': True, 'Yield': True, 'Crop_Category': True, 'weather_available': True, 'tmin_mean_c': True, 'tmax_mean_c': True, 'precipitation_total_mm': True}

Setup and schema validation passed.


In [2]:

# Keep only individual-crop candidates
risk_df = df[
    df["Crop_Category"].eq("Individual Crop Candidate")
].copy()

# Normalize key fields and numeric columns
key_cols = ["State", "District", "Crop", "Season"]

for col in key_cols:
    risk_df[col] = risk_df[col].astype("string").str.strip()

risk_df["start_year"] = pd.to_numeric(
    risk_df["start_year"], errors="coerce"
)
risk_df["Yield"] = pd.to_numeric(
    risk_df["Yield"], errors="coerce"
)

# Exclude aggregate "Total" seasons and unknown season labels
named_seasons = [
    "Kharif", "Rabi", "Summer",
    "Whole Year", "Autumn", "Winter"
]

risk_df = risk_df[
    risk_df["Season"].isin(named_seasons)
    & risk_df["start_year"].notna()
    & risk_df["Yield"].notna()
    & np.isfinite(risk_df["Yield"])
    & (risk_df["Yield"] > 0)
    & risk_df[key_cols].notna().all(axis=1)
].copy()

risk_df["start_year"] = risk_df["start_year"].astype(int)

# Check whether each analysis key has one record per year
record_key = key_cols + ["start_year"]
duplicate_count = int(risk_df.duplicated(record_key, keep=False).sum())

print("Prepared rows:", len(risk_df))
print("Unique crops:", risk_df["Crop"].nunique())
print("Unique districts:", risk_df["District"].nunique())
print("Years:", risk_df["start_year"].min(), "to",
      risk_df["start_year"].max())
print("Season counts:")
print(risk_df["Season"].value_counts())
print("Rows participating in duplicate keys:", duplicate_count)
print("Missing yield:", risk_df["Yield"].isna().sum())


Prepared rows: 90893
Unique crops: 13
Unique districts: 748
Years: 2013 to 2024
Season counts:
Season
Kharif    45783
Rabi      34178
Summer    10932
Name: count, dtype: Int64
Rows participating in duplicate keys: 0
Missing yield: 0


In [3]:

group_cols = ["State", "District", "Crop", "Season"]

risk_df = risk_df.sort_values(
    group_cols + ["start_year"]
).copy()

# Historical baseline: previous 3 observations, excluding the current year.
risk_df["historical_median_yield"] = (
    risk_df.groupby(group_cols)["Yield"]
    .transform(
        lambda s: s.shift(1).rolling(
            window=3, min_periods=2
        ).median()
    )
)

# Count how many prior observations support each baseline.
risk_df["historical_observation_count"] = (
    risk_df.groupby(group_cols)["Yield"]
    .transform(
        lambda s: s.shift(1).rolling(
            window=3, min_periods=2
        ).count()
    )
)

risk_df["yield_vs_historical_pct"] = (
    (
        risk_df["Yield"] - risk_df["historical_median_yield"]
    )
    / risk_df["historical_median_yield"].replace(0, np.nan)
) * 100

baseline_df = risk_df[
    risk_df["historical_median_yield"].notna()
].copy()

print("Total prepared observations:", len(risk_df))
print("Observations with historical baseline:", len(baseline_df))
print(
    "Observations without sufficient history:",
    risk_df["historical_median_yield"].isna().sum()
)
print("\nHistorical deviation (%) summary:")
print(baseline_df["yield_vs_historical_pct"].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
).round(2))


Total prepared observations: 90893
Observations with historical baseline: 69693
Observations without sufficient history: 21200

Historical deviation (%) summary:
count      69693.00
mean          88.21
std        20863.25
min          -99.92
5%           -39.83
25%           -9.09
50%            1.27
75%           17.13
95%           72.16
max      5507614.29
Name: yield_vs_historical_pct, dtype: float64


In [4]:

# Compare current yield with its historical median.
baseline_df["yield_to_baseline_ratio"] = (
    baseline_df["Yield"] / baseline_df["historical_median_yield"]
)

print("Yield-to-baseline ratio summary:")
print(
    baseline_df["yield_to_baseline_ratio"]
    .replace([np.inf, -np.inf], np.nan)
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .round(3)
)

print("\nSmallest historical baselines:")
print(
    baseline_df.nsmallest(10, "historical_median_yield")[
        ["State", "District", "Crop", "Season", "start_year",
         "Yield", "historical_median_yield",
         "yield_vs_historical_pct"]
    ].to_string(index=False)
)

print("\nLargest percentage deviations:")
print(
    baseline_df.nlargest(10, "yield_vs_historical_pct")[
        ["State", "District", "Crop", "Season", "start_year",
         "Yield", "historical_median_yield",
         "yield_vs_historical_pct"]
    ].to_string(index=False)
)


Yield-to-baseline ratio summary:
count    69693.000
mean         1.882
std        208.633
min          0.001
1%           0.329
5%           0.602
25%          0.909
50%          1.013
75%          1.171
95%          1.722
99%          2.683
max      55077.143
Name: yield_to_baseline_ratio, dtype: float64

Smallest historical baselines:
                      State       District          Crop Season  start_year    Yield  historical_median_yield  yield_vs_historical_pct
Andaman And Nicobar Islands South Andamans          Urad   Rabi        2022    492.0                      7.0             6.928571e+03
Andaman And Nicobar Islands South Andamans          Urad   Rabi        2023 385540.0                      7.0             5.507614e+06
Andaman And Nicobar Islands South Andamans          Urad   Rabi        2021      7.0                     20.0            -6.500000e+01
          Jammu And Kashmir         Kathua        Lentil   Rabi        2022     50.0                     50.0            

In [5]:

# Work on a separate copy so the original data remains unchanged.
risk_indicators = baseline_df.copy()

ratio = risk_indicators["yield_to_baseline_ratio"]

risk_indicators["extreme_yield_deviation_flag"] = (
    (ratio < 0.1) | (ratio > 5)
)

risk_indicators["yield_risk_indicator"] = np.select(
    [
        ratio < 0.70,
        ratio < 0.85,
        ratio <= 1.15,
        ratio <= 5,
        ratio > 5
    ],
    [
        "High Concern",
        "Moderate Concern",
        "Within Historical Range",
        "Above Historical Range",
        "Extreme Deviation - Review Data"
    ],
    default="Unclassified"
)

print("Risk-indicator counts:")
print(risk_indicators["yield_risk_indicator"].value_counts())

print("\nExtreme-deviation flags:")
print(
    risk_indicators["extreme_yield_deviation_flag"]
    .value_counts()
)

print("\nExtreme records by crop:")
print(
    risk_indicators.loc[
        risk_indicators["extreme_yield_deviation_flag"]
    ]
    .groupby("Crop")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print("\nExample records requiring review:")
print(
    risk_indicators.loc[
        risk_indicators["extreme_yield_deviation_flag"],
        ["State", "District", "Crop", "Season", "start_year",
         "Yield", "historical_median_yield",
         "yield_to_baseline_ratio"]
    ]
    .sort_values("yield_to_baseline_ratio", ascending=False)
    .head(10)
    .to_string(index=False)
)


Risk-indicator counts:
yield_risk_indicator
Within Historical Range            37772
Above Historical Range             18926
Moderate Concern                    6968
High Concern                        5943
Extreme Deviation - Review Data       84
Name: count, dtype: int64

Extreme-deviation flags:
extreme_yield_deviation_flag
False    69543
True       150
Name: count, dtype: int64

Extreme records by crop:
Crop
Maize            21
Jowar            20
Small Millets    19
Tur              19
Urad             18
Rice             16
Moong            11
Gram              8
Lentil            6
Bajra             5
dtype: int64

Example records requiring review:
                                       State               District          Crop Season  start_year    Yield  historical_median_yield  yield_to_baseline_ratio
                 Andaman And Nicobar Islands         South Andamans          Urad   Rabi        2023 385540.0                      7.0             55077.142857
               

In [6]:

# Fix the category precedence so extreme values are classified first.
ratio = risk_indicators["yield_to_baseline_ratio"]

risk_indicators["yield_risk_indicator"] = np.select(
    [
        (ratio < 0.1) | (ratio > 5),
        ratio < 0.70,
        ratio < 0.85,
        ratio <= 1.15,
        ratio <= 5
    ],
    [
        "Extreme Deviation - Review Data",
        "High Concern",
        "Moderate Concern",
        "Within Historical Range",
        "Above Historical Range"
    ],
    default="Unclassified"
)

print("Corrected risk-indicator counts:")
print(risk_indicators["yield_risk_indicator"].value_counts())

print("\nHigh and moderate concern by crop:")
print(
    risk_indicators[
        risk_indicators["yield_risk_indicator"].isin(
            ["High Concern", "Moderate Concern"]
        )
    ]
    .groupby("Crop")
    .agg(
        flagged_observations=("Yield", "size"),
        median_yield=("Yield", "median")
    )
    .sort_values("flagged_observations", ascending=False)
    .round(2)
)

print("\nRisk indicator distribution by season:")
print(
    pd.crosstab(
        risk_indicators["Season"],
        risk_indicators["yield_risk_indicator"]
    )
)


Corrected risk-indicator counts:
yield_risk_indicator
Within Historical Range            37772
Above Historical Range             18926
Moderate Concern                    6968
High Concern                        5877
Extreme Deviation - Review Data      150
Name: count, dtype: int64

High and moderate concern by crop:
               flagged_observations  median_yield
Crop                                             
Moong                          1851         397.0
Maize                          1801        1949.0
Urad                           1578         429.0
Rice                           1462        1702.0
Tur                            1164         660.0
Jowar                          1091         769.0
Gram                            805         802.0
Bajra                           649         927.0
Lentil                          562         650.0
Wheat                           560        1568.5
Ragi                            508         895.0
Small Millets                

In [7]:

# Check whether weather fields are actually available for each record.
weather_cols = [
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm"
]

for col in weather_cols:
    risk_indicators[col] = pd.to_numeric(
        risk_indicators[col], errors="coerce"
    )

risk_indicators["weather_fields_available"] = (
    risk_indicators[weather_cols].notna().all(axis=1)
)

risk_indicators["weather_available"] = (
    risk_indicators["weather_available"]
    .astype("string")
    .str.lower()
    .eq("true")
)

# Summarize risk indicators by weather availability.
weather_risk_summary = (
    risk_indicators
    .groupby("weather_available", dropna=False)
    .agg(
        observations=("Yield", "size"),
        high_concern=(
            "yield_risk_indicator",
            lambda s: s.eq("High Concern").sum()
        ),
        moderate_concern=(
            "yield_risk_indicator",
            lambda s: s.eq("Moderate Concern").sum()
        ),
        extreme_deviation=(
            "yield_risk_indicator",
            lambda s: s.eq("Extreme Deviation - Review Data").sum()
        ),
        complete_weather_fields=("weather_fields_available", "sum")
    )
)

weather_risk_summary["high_concern_pct"] = (
    weather_risk_summary["high_concern"]
    / weather_risk_summary["observations"] * 100
)

weather_risk_summary["moderate_concern_pct"] = (
    weather_risk_summary["moderate_concern"]
    / weather_risk_summary["observations"] * 100
)

print("Yield indicators by weather availability:")
print(weather_risk_summary.round(2))

print("\nWeather alignment metadata:")
print(
    df["weather_alignment_method"]
    .value_counts(dropna=False)
    if "weather_alignment_method" in df.columns
    else "Alignment metadata column not found."
)

print("\nVerified full-season alignment:")
print(
    df["weather_full_season_alignment_verified"]
    .value_counts(dropna=False)
    if "weather_full_season_alignment_verified" in df.columns
    else "Full-season verification column not found."
)


Yield indicators by weather availability:
                   observations  high_concern  moderate_concern  \
weather_available                                                 
False                      9692           909               967   
True                      60001          4968              6001   

                   extreme_deviation  complete_weather_fields  \
weather_available                                               
False                             29                        0   
True                             121                    60001   

                   high_concern_pct  moderate_concern_pct  
weather_available                                          
False                          9.38                  9.98  
True                           8.28                 10.00  

Weather alignment metadata:
weather_alignment_method
STARTING_CALENDAR_YEAR_PROXY    376272
Name: count, dtype: int64

Verified full-season alignment:
weather_full_season_alignment_verifi

In [8]:

weather_analysis = risk_indicators[
    risk_indicators["weather_fields_available"]
].copy()

weather_summary = (
    weather_analysis
    .groupby("yield_risk_indicator")
    .agg(
        observations=("Yield", "size"),
        median_yield=("Yield", "median"),
        median_historical_yield=("historical_median_yield", "median"),
        median_min_temperature_c=("tmin_mean_c", "median"),
        median_max_temperature_c=("tmax_mean_c", "median"),
        median_annual_precipitation_mm=("precipitation_total_mm", "median")
    )
    .sort_values("observations", ascending=False)
)

print("Weather summaries by yield indicator:")
print(weather_summary.round(2))

print("\nWeather data quality checks:")
print(
    weather_analysis[
        weather_cols
    ].describe(percentiles=[0.05, 0.50, 0.95]).round(2)
)

print("\nRows used:", len(weather_analysis))
print(
    "Full-season alignment verified:",
    bool(df["weather_full_season_alignment_verified"].any())
)


Weather summaries by yield indicator:
                                 observations  median_yield  \
yield_risk_indicator                                          
Within Historical Range                 32569        1385.0   
Above Historical Range                  16342        1400.0   
Moderate Concern                         6001         973.0   
High Concern                             4968         578.0   
Extreme Deviation - Review Data           121        1000.0   

                                 median_historical_yield  \
yield_risk_indicator                                       
Within Historical Range                           1386.0   
Above Historical Range                             981.5   
Moderate Concern                                  1235.0   
High Concern                                      1090.0   
Extreme Deviation - Review Data                    500.0   

                                 median_min_temperature_c  \
yield_risk_indicator                  

In [9]:

# Summarize risk indicators for each district-crop-season combination.
risk_summary = (
    risk_indicators
    .groupby(
        ["State", "District", "Crop", "Season"],
        as_index=False
    )
    .agg(
        observations=("Yield", "size"),
        first_year=("start_year", "min"),
        last_year=("start_year", "max"),
        median_yield=("Yield", "median"),
        median_historical_yield=("historical_median_yield", "median"),
        high_concern_count=(
            "yield_risk_indicator",
            lambda s: s.eq("High Concern").sum()
        ),
        moderate_concern_count=(
            "yield_risk_indicator",
            lambda s: s.eq("Moderate Concern").sum()
        ),
        extreme_deviation_count=(
            "yield_risk_indicator",
            lambda s: s.eq("Extreme Deviation - Review Data").sum()
        )
    )
)

risk_summary["high_concern_pct"] = (
    risk_summary["high_concern_count"]
    / risk_summary["observations"] * 100
)

risk_summary["moderate_concern_pct"] = (
    risk_summary["moderate_concern_count"]
    / risk_summary["observations"] * 100
)

# Avoid assigning a definitive risk label when history is very limited.
risk_summary["evidence_status"] = np.select(
    [
        risk_summary["observations"] < 3,
        risk_summary["observations"] < 5
    ],
    [
        "Insufficient History",
        "Limited History"
    ],
    default="Historical Pattern Available"
)

print("District-crop-season groups:", len(risk_summary))
print("\nEvidence status:")
print(risk_summary["evidence_status"].value_counts())

print("\nGroups with the highest high-concern rates (at least 5 observations):")
print(
    risk_summary[risk_summary["observations"] >= 5]
    .sort_values(
        ["high_concern_pct", "observations"],
        ascending=[False, False]
    )
    .head(15)[
        ["State", "District", "Crop", "Season", "observations",
         "first_year", "last_year", "high_concern_count",
         "high_concern_pct", "moderate_concern_pct", "evidence_status"]
    ]
    .round(2)
    .to_string(index=False)
)


District-crop-season groups: 9236

Evidence status:
evidence_status
Historical Pattern Available    7254
Insufficient History            1105
Limited History                  877
Name: count, dtype: int64

Groups with the highest high-concern rates (at least 5 observations):
                      State       District          Crop Season  observations  first_year  last_year  high_concern_count  high_concern_pct  moderate_concern_pct              evidence_status
                     Kerala         Idukki         Wheat   Rabi             6        2016       2023                   4             66.67                   0.0 Historical Pattern Available
             Andhra Pradesh  Y.S.R. Kadapa         Moong Kharif            10        2015       2024                   6             60.00                   0.0 Historical Pattern Available
                 Tamil Nadu   Nagapattinam          Urad   Rabi            10        2015       2024                   6             60.00                

In [10]:

# Measure whether high-concern observations occur across multiple years.
high_concern_df = risk_indicators[
    risk_indicators["yield_risk_indicator"].eq("High Concern")
].copy()

high_concern_persistence = (
    high_concern_df
    .groupby(["State", "District", "Crop", "Season"], as_index=False)
    .agg(
        high_concern_years=("start_year", "nunique"),
        first_high_concern_year=("start_year", "min"),
        last_high_concern_year=("start_year", "max")
    )
)

risk_summary = risk_summary.merge(
    high_concern_persistence,
    on=["State", "District", "Crop", "Season"],
    how="left"
)

risk_summary["high_concern_years"] = (
    risk_summary["high_concern_years"].fillna(0).astype(int)
)

# A descriptive persistence label; not a forecast.
risk_summary["persistence_indicator"] = np.select(
    [
        risk_summary["high_concern_years"] >= 3,
        risk_summary["high_concern_years"] == 2,
        risk_summary["high_concern_years"] == 1
    ],
    [
        "Repeated in 3+ Years",
        "Repeated in 2 Years",
        "Observed in 1 Year"
    ],
    default="No High-Concern Years"
)

print("Persistence indicator counts:")
print(risk_summary["persistence_indicator"].value_counts())

print("\nRepeated high-concern patterns (at least 5 observations):")
print(
    risk_summary[
        (risk_summary["observations"] >= 5)
        & (risk_summary["high_concern_years"] >= 3)
    ]
    .sort_values(
        ["high_concern_years", "high_concern_pct", "observations"],
        ascending=False
    )
    .head(15)[
        ["State", "District", "Crop", "Season", "observations",
         "high_concern_count", "high_concern_pct",
         "high_concern_years", "first_high_concern_year",
         "last_high_concern_year", "evidence_status"]
    ]
    .round(2)
    .to_string(index=False)
)


Persistence indicator counts:
persistence_indicator
No High-Concern Years    5566
Observed in 1 Year       2074
Repeated in 2 Years      1108
Repeated in 3+ Years      488
Name: count, dtype: int64

Repeated high-concern patterns (at least 5 observations):
                      State       District  Crop Season  observations  high_concern_count  high_concern_pct  high_concern_years  first_high_concern_year  last_high_concern_year              evidence_status
             Andhra Pradesh  Y.S.R. Kadapa Moong Kharif            10                   6             60.00                   6                   2015.0                  2024.0 Historical Pattern Available
                 Tamil Nadu   Nagapattinam  Urad   Rabi            10                   6             60.00                   6                   2015.0                  2022.0 Historical Pattern Available
             Andhra Pradesh  Ananthapuramu  Gram   Rabi            10                   5             50.00                  

In [11]:

import json

# Observation-level results
observation_output = OUTPUT_DIR / "yield_risk_indicators.csv"
risk_indicators.to_csv(observation_output, index=False)

# District-crop-season summaries
summary_output = OUTPUT_DIR / "district_crop_risk_summary.csv"
risk_summary.to_csv(summary_output, index=False)

# Extreme deviations requiring source-data review
extreme_output = OUTPUT_DIR / "extreme_yield_deviations_review.csv"
risk_indicators[
    risk_indicators["extreme_yield_deviation_flag"]
].sort_values(
    "yield_to_baseline_ratio", ascending=False
).to_csv(extreme_output, index=False)

metadata = {
    "module": "Agricultural Risk Intelligence",
    "source_dataset": str(DATA_PATH),
    "analysis_period": [
        int(risk_df["start_year"].min()),
        int(risk_df["start_year"].max())
    ],
    "prepared_observations": int(len(risk_df)),
    "observations_with_historical_baseline": int(len(baseline_df)),
    "district_crop_season_groups": int(len(risk_summary)),
    "risk_definition": {
        "high_concern": "Current yield < 70% of previous-three-observation median",
        "moderate_concern": "Current yield >= 70% and < 85% of historical median",
        "within_historical_range": "Current yield >= 85% and <= 115% of historical median",
        "above_historical_range": "Current yield > 115% and <= 5 times historical median",
        "extreme_deviation_review": "Yield ratio < 0.1 or > 5"
    },
    "persistence_definition": "Count of distinct years classified as High Concern",
    "limitations": [
        "Exploratory historical indicators, not calibrated future-risk probabilities.",
        "Thresholds are heuristic and not validated agronomic thresholds.",
        "Extreme yield deviations require source-data and unit review.",
        "Weather associations are descriptive, not causal.",
        "Weather uses an unverified starting-calendar-year proxy.",
        "No full crop-season weather alignment has been verified.",
        "Missing weather data is unavailable, not evidence of low risk."
    ],
    "deployment_status": "EXPLORATORY — NOT VALIDATED FOR FARMER-FACING RISK WARNINGS"
}

metadata_output = OUTPUT_DIR / "agricultural_risk_metadata.json"
with open(metadata_output, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved outputs:")
for path in [
    observation_output,
    summary_output,
    extreme_output,
    metadata_output
]:
    print(f"{path.name}: {'OK' if path.exists() else 'MISSING'}")

print("\nObservation rows:", len(risk_indicators))
print("Summary groups:", len(risk_summary))
print(
    "Extreme records:",
    int(risk_indicators["extreme_yield_deviation_flag"].sum())
)
print("Deployment status:", metadata["deployment_status"])


Saved outputs:
yield_risk_indicators.csv: OK
district_crop_risk_summary.csv: OK
extreme_yield_deviations_review.csv: OK
agricultural_risk_metadata.json: OK

Observation rows: 69693
Summary groups: 9236
Extreme records: 150
Deployment status: EXPLORATORY — NOT VALIDATED FOR FARMER-FACING RISK WARNINGS



# Final Conclusion — Agricultural Risk Intelligence

## 1. Objective
This notebook investigated historical agricultural yield anomalies and developed interpretable risk indicators using the existing integrated APY and annual district-level weather dataset.

## 2. Work Completed
- Prepared valid individual-crop observations, excluding aggregate totals and unverified grouping categories.
- Established historical yield baselines using up to three previous observations for the same state, district, crop, and season.
- Classified observations into high concern, moderate concern, within historical range, above historical range, and extreme deviation requiring review.
- Summarized historical yield indicators across district–crop–season combinations.
- Examined whether high-concern indicators repeated across multiple years.
- Compared yield-indicator categories with available annual weather summaries.
- Exported observation-level results, summary tables, extreme-deviation records, and metadata.

## 3. Key Results
- Prepared individual-crop observations: 90,893.
- Observations with sufficient historical baselines: 69,693.
- District–crop–season combinations summarized: 9,236.
- High-concern observations: 5,877.
- Moderate-concern observations: 6,968.
- Extreme yield deviations flagged for review: 150.
- Combinations with high-concern observations in at least three distinct years: 488.

## 4. Risk Indicator Definitions
- **High Concern:** Current yield below 70% of the historical median.
- **Moderate Concern:** Current yield from 70% up to, but not including, 85% of the historical median.
- **Within Historical Range:** Current yield from 85% through 115% of the historical median.
- **Above Historical Range:** Current yield above 115% and no more than five times the historical median.
- **Extreme Deviation — Review Data:** Yield ratio below 0.1 or above 5.

These thresholds are exploratory heuristics, not validated agronomic thresholds or official crop-failure definitions.

## 5. Generated Artifacts
The following files were saved in `models/agricultural_risk_intelligence/`:

- `yield_risk_indicators.csv`
- `district_crop_risk_summary.csv`
- `extreme_yield_deviations_review.csv`
- `agricultural_risk_metadata.json`

## 6. Limitations
- Historical yield anomalies do not establish their causes or predict future crop losses.
- Extreme observations may reflect genuine changes, source-data errors, or inconsistent units and require investigation.
- Groups with limited historical observations have weaker evidence.
- Weather comparisons are descriptive and do not establish causation.
- Weather uses an unverified starting-calendar-year proxy; full crop-season alignment has not been verified.
- Annual weather summaries cannot support short-term weather forecasting.
- The indicators are not calibrated probabilities and must not be presented as validated farmer-facing risk warnings.

## 7. Final Status

**Notebook Status:** COMPLETE

**Historical Yield Risk Indicators:** Generated

**Repeated Historical Patterns:** Identified

**Extreme Deviations:** Flagged for review

**Weather-Based Causal Claims:** Not established

**Deployment Status:** EXPLORATORY — NOT VALIDATED FOR FARMER-FACING RISK WARNINGS

Notebook 20 establishes a foundation for historical agricultural risk investigation in BharatCrop-Intelligence. Further data-quality checks, agronomic validation, and out-of-time evaluation are required before these indicators can support farmer-facing warnings.
